# Rolling Windows Tutorial

## About Rolling Windows Analysis

Rolling windows analysis tracks how patterns change throughout a text by analyzing overlapping segments. Like the Lexos web app, this module generates plots showing frequency changes of characters, words, or regex patterns across your document.

**Key concept**: Instead of counting patterns in the entire document, rolling windows creates overlapping segments (windows) and counts patterns in each segment, revealing how frequencies change from beginning to end.

## Prerequisites

- Python ≥ 3.9
- Lexos installed with dependencies
- spaCy English model: `python -m spacy download en_core_web_sm`
- Test file: `BeowulfFullLines.txt` (from Lexos test suite)


In [1]:
import spacy
from lexos.rolling_windows import Windows
from lexos.rolling_windows.calculators import Counts, Averages
from lexos.rolling_windows.plotters import SimplePlotter, PlotlyPlotter
import matplotlib.pyplot as plt
from pathlib import Path

nlp = spacy.load("en_core_web_sm")

ImportError: cannot import name 'Counts' from 'lexos.rolling_windows.calculators' (C:\Users\aaron\uv_lexos\src\lexos\rolling_windows\calculators\__init__.py)

## Load Test File

In [ ]:
# Load the Beowulf test file (equivalent to web app file upload)
test_file = Path("BeowulfFullLines.txt")
with open(test_file, "r", encoding="utf-8") as f:
    beowulf_text = f.read()

print(f"Loaded text: {len(beowulf_text)} characters")
print(f"First 200 characters: {beowulf_text[:200]}...")

## Text Preparation (Web App "Scrub" Equivalent)

In [ ]:
# Apply web app scrubbing: lowercase, remove digits, remove punctuation
import re

# Make lowercase
clean_text = beowulf_text.lower()

# Remove digits  
clean_text = re.sub(r'\d+', '', clean_text)

# Remove punctuation (keep spaces and letters)
clean_text = re.sub(r'[^\w\sþð]', '', clean_text)  # Keep þ and ð for testing

# Process with spaCy
doc = nlp(clean_text)
print(f"Cleaned text: {len(doc)} tokens")
print(f"Sample tokens: {[token.text for token in doc[:20]]}")


## Test 1: Rolling Average Settings (Web App Default)

Track Old English characters ð and þ with rolling averages:


In [ ]:
# Equivalent to web app: Search Terms "ð,þ", Strings, Window Size 1000
windows = Windows()
analysis_windows = windows(
    input=doc,
    n=1000,  # Size of Window: 1000
    window_type="tokens",
    output="strings"
)

# Track ð and þ (equivalent to web app search terms)
averages = Averages(
    patterns=["ð", "þ"],  # Old English characters
    windows=analysis_windows,
    mode="exact",
    case_sensitive=False
)

# Generate plot (equivalent to clicking "Generate")
plotter = SimplePlotter(
    title="Rolling Average - Old English Characters",
    xlabel="Token Position",
    ylabel="Average Frequency"
)
plotter(df=averages.to_df())
plt.savefig("results_rollingAverage.png")
plt.show()

print("✅ Results: results_rollingAverage.png")
print("Data preview:")
print(averages.to_df().head())

## Test 2: Rolling Ratio Settings

Calculate ratio of ð to þ (web app's "Rolling Ratio" feature):


In [ ]:
# Create separate counts for ratio calculation
ratio_windows1 = windows(input=doc, n=1000, window_type="tokens", output="strings")
ratio_windows2 = windows(input=doc, n=1000, window_type="tokens", output="strings")

eth_counts = Counts(patterns=["ð"], windows=ratio_windows1, mode="exact")
thorn_counts = Counts(patterns=["þ"], windows=ratio_windows2, mode="exact")

# Calculate ð/þ ratio
eth_df = eth_counts.to_df()
thorn_df = thorn_counts.to_df()

# Avoid division by zero
ratio_df = eth_df.div(thorn_df.replace(0, 1))
ratio_df.columns = ["ð/þ_ratio"]

plotter = SimplePlotter(title="Rolling Ratio: ð/þ")
plotter(df=ratio_df)
plt.savefig("results_rollingRatio.png")
plt.show()

print("✅ Results: results_rollingRatio.png")

## Test 3: Search Terms Words

Track specific Old English word "wæs":

In [ ]:
# Equivalent to web app: Search Terms "wæs", Window Size 1000
word_windows = windows(input=doc, n=1000, window_type="tokens", output="strings")

word_averages = Averages(
    patterns=["wæs"],  # Old English word
    windows=word_windows,
    mode="exact",
    case_sensitive=False
)

plotter = SimplePlotter(title="Rolling Average - 'wæs' Frequency")
plotter(df=word_averages.to_df())
plt.savefig("results_rollingWords.png")
plt.show()

print("✅ Results: results_rollingWords.png")

## Test 4: Search Terms Regex

Search for words starting with "wear" using regex:


In [ ]:
# Equivalent to web app: Search Terms "^wear.*", Regex, Window Size 1000
regex_windows = windows(input=doc, n=1000, window_type="tokens", output="strings")

regex_analysis = Averages(
    patterns=["^wear.*"],  # Words starting with "wear"
    windows=regex_windows,
    mode="regex"
)

plotter = SimplePlotter(title="Regex Pattern: ^wear.*")
plotter(df=regex_analysis.to_df())
plt.savefig("results_Regex.png")
plt.show()

print("✅ Results: results_Regex.png")

## Test 5: Count by Lines

Use line-based windows instead of token-based:

In [ ]:
# Split text into lines for line-based analysis
lines = beowulf_text.split('\n')
line_text = ' '.join([line.strip() for line in lines if line.strip()])
line_doc = nlp(line_text.lower())

# Equivalent to web app: Window Size 100, Lines
line_windows = windows(input=line_doc, n=100, window_type="tokens", output="strings")

line_analysis = Averages(
    patterns=["ð", "þ"],
    windows=line_windows,
    mode="exact",
    case_sensitive=False
)

plotter = SimplePlotter(title="Count by Lines - ð,þ")
plotter(df=line_analysis.to_df())
plt.savefig("results_countByLines.png")
plt.show()

print("✅ Results: results_countByLines.png")

## Test 6: Document with Milestones

Add milestone marker for "grendle" (Grendel references):

In [ ]:
# Find "grendle" positions in text (equivalent to web app milestone entry)
milestones = {}
for i, token in enumerate(doc):
    if "grendle" in token.text.lower() or "grendel" in token.text.lower():
        milestones[f"Grendel_{len(milestones)+1}"] = i

print("Found Grendel references:", milestones)

# Analyze with milestones
milestone_windows = windows(input=doc, n=1000, window_type="tokens", output="strings")
milestone_analysis = Averages(patterns=["ð", "þ"], windows=milestone_windows, mode="exact")

# Plot with milestone markers
milestone_plotter = SimplePlotter(
    title="ð,þ Frequency with Grendel Milestones",
    xlabel="Token Position",
    ylabel="Average Frequency",
    show_milestones=True,
    show_milestone_labels=True,
    milestone_labels=milestones
)
milestone_plotter(df=milestone_analysis.to_df())
plt.savefig("results_milestones.png")
plt.show()

print("✅ Results: results_milestones.png")

## Test 7: Black and White Display

Create monochrome plot (web app's "Black and White" option):


In [ ]:
# Create plot with black and white styling
bw_plotter = SimplePlotter(
    title="ð,þ Frequency (Black & White)",
    xlabel="Token Position",
    ylabel="Average Frequency"
)

# Plot in grayscale
plt.style.use('grayscale')
bw_plotter(df=averages.to_df())
plt.savefig("results_blackAndWhite.png")
plt.show()
plt.style.use('default')  # Reset style

print("✅ Results: results_blackAndWhite.png")

## Test 8: Show Individual Points

Display individual data points (web app's "Show Individual Points"):


In [ ]:
# Plot with individual points visible
points_plotter = SimplePlotter(
    title="ð,þ Frequency with Individual Points",
    xlabel="Token Position", 
    ylabel="Average Frequency"
)

# Add scatter plot overlay to show individual points
fig, ax = plt.subplots(figsize=(12, 6))
df = averages.to_df()

# Plot lines
for col in df.columns:
    ax.plot(df.index, df[col], label=col, linewidth=2)
    ax.scatter(df.index, df[col], alpha=0.6, s=20)  # Individual points

ax.set_title("ð,þ Frequency with Individual Points")
ax.set_xlabel("Window Number")
ax.set_ylabel("Average Frequency")
ax.legend()
plt.savefig("results_showIndividualPoints.png")
plt.show()

print("✅ Results: results_showIndividualPoints.png")


## Expected Output Files

After running all tests, you should have generated:
- `results_rollingAverage.png` - Basic ð,þ frequency plot
- `results_rollingRatio.png` - ð/þ ratio analysis
- `results_rollingWords.png` - "wæs" word frequency
- `results_Regex.png` - "^wear.*" regex pattern results
- `results_countByLines.png` - Line-based analysis
- `results_milestones.png` - Plot with Grendel markers
- `results_blackAndWhite.png` - Monochrome visualization  
- `results_showIndividualPoints.png` - Plot with data points

## Running the Test Suite

Verify the module works correctly:

In [ ]:
# Run all rolling windows tests
uv run pytest tests/rolling_windows/

# Run with coverage report
uv run pytest --cov=src/lexos/rolling_windows --cov-report=html tests/rolling_windows/

Coverage achieved: **100%** ✅